# 03. Eksperimen Utama: Class Weighting (Skenario A), Risk Ranking (Skenario B), & Cost-Aware Recall@20% LOC (Skenario C)

Notebook ini menjalankan seluruh eksperimen utama:
- **Baseline**: Logistic Regression & Random Forest (`class_weight=None`)
- **Skenario A**: Logistic Regression & Random Forest (`class_weight='balanced'`)
- **Skenario B**: Pemeringkatan modul berdasarkan `predict_proba` kelas defective
- **Skenario C**: Evaluasi `Recall@20% LOC` dan perbandingan dengan **Random Baseline (0.20)**
- Output otomatis disimpan ke `../results/metrics.csv` dan grafik ke `../figures/`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

sys.path.append(str(Path("..").resolve()))
from src.preprocessing import load_nasa_arff, clean_data, split_train_val_test, build_model_pipeline
from src.evaluation import (
    calculate_metrics,
    calculate_recall_at_20_percent_loc,
    compute_loc_recall_curve,
    get_pr_curve_points,
)

data_dir = Path("../data")
results_dir = Path("../results")
figures_dir = Path("../figures")
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

datasets = ["JM1.arff", "PC4.arff", "MC1.arff"]
all_results = []
curve_store = {}

for fname in datasets:
    ds_name = fname.replace(".arff", "")
    df, target_col = load_nasa_arff(data_dir / fname)
    X, y = clean_data(df, target_col=target_col)

    X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
        X, y, train_size=0.60, val_size=0.20, test_size=0.20, random_state=42
    )

    models = {
        "Baseline_LR": build_model_pipeline(
            LogisticRegression(class_weight=None, max_iter=1000, random_state=42), use_scaler=True
        ),
        "Baseline_RF": build_model_pipeline(
            RandomForestClassifier(n_estimators=100, class_weight=None, random_state=42), use_scaler=False
        ),
        "Weighted_LR": build_model_pipeline(
            LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42), use_scaler=True
        ),
        "Weighted_RF": build_model_pipeline(
            RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42), use_scaler=False
        ),
    }

    loc_val = X_val["LOC_TOTAL"].values
    loc_test = X_test["LOC_TOTAL"].values
    loc_test_safe = np.where(loc_test <= 0, 1.0, loc_test)
    curve_store[ds_name] = {"y_test": y_test.values, "loc_test": loc_test, "models": {}}

    for name, model in models.items():
        model.fit(X_train, y_train)

        val_prob = model.predict_proba(X_val)[:, 1]
        val_pred = model.predict(X_val)
        val_m = calculate_metrics(y_val.values, val_pred, val_prob)
        val_rec20_prob = calculate_recall_at_20_percent_loc(y_val.values, val_prob, loc_val)

        test_prob = model.predict_proba(X_test)[:, 1]
        test_pred = model.predict(X_test)
        test_m = calculate_metrics(y_test.values, test_pred, test_prob)

        # Skenario B & C: Ranking probabilitas & Density (Prob/LOC)
        rec20_prob = calculate_recall_at_20_percent_loc(y_test.values, test_prob, loc_test)
        rec20_density = calculate_recall_at_20_percent_loc(y_test.values, test_prob / loc_test_safe, loc_test)

        curve_store[ds_name]["models"][name] = test_prob

        all_results.append({
            "Dataset": ds_name,
            "Model": name,
            "Val_PR_AUC": round(val_m["pr_auc"], 4),
            "Val_Precision": round(val_m["precision"], 4),
            "Val_Recall": round(val_m["recall"], 4),
            "Val_F1": round(val_m["f1"], 4),
            "Val_Recall20_Prob": round(val_rec20_prob, 4),
            "Test_PR_AUC": round(test_m["pr_auc"], 4),
            "Test_Precision": round(test_m["precision"], 4),
            "Test_Recall": round(test_m["recall"], 4),
            "Test_F1": round(test_m["f1"], 4),
            "Recall20_Prob": round(rec20_prob, 4),
            "Recall20_Density": round(rec20_density, 4),
            "Random_Baseline_LOC": 0.2000
        })

df_res = pd.DataFrame(all_results)
df_res.to_csv(results_dir / "metrics.csv", index=False)
print(df_res.to_string(index=False))

Dataset       Model  Val_PR_AUC  Val_Precision  Val_Recall  Val_F1  Val_Recall20_Prob  Test_PR_AUC  Test_Precision  Test_Recall  Test_F1  Recall20_Prob  Recall20_Density  Random_Baseline_LOC
    JM1 Baseline_LR      0.3783         0.4627      0.0928  0.1546             0.0689       0.3830          0.5741       0.0925   0.1594         0.1164            0.3552                  0.2
    JM1 Baseline_RF      0.4243         0.5593      0.1976  0.2920             0.1287       0.3657          0.4919       0.1821   0.2658         0.1552            0.3493                  0.2
    JM1 Weighted_LR      0.3758         0.3462      0.5090  0.4121             0.0689       0.3841          0.3453       0.5164   0.4139         0.1104            0.3582                  0.2
    JM1 Weighted_RF      0.4270         0.4762      0.4192  0.4459             0.1527       0.3674          0.4223       0.3731   0.3962         0.1522            0.3701                  0.2
    PC4 Baseline_LR      0.5864         0.722

In [2]:
# --- GRAFIK 1: Perbandingan Test PR-AUC ---
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=df_res, x="Dataset", y="Test_PR_AUC", hue="Model", ax=ax)
ax.set_title("Perbandingan Test PR-AUC: Baseline vs Class Weighting")
ax.set_ylim(0, 0.85)
plt.tight_layout()
plt.savefig(figures_dir / "pr_auc_comparison.png", dpi=300)
plt.close(fig)

# --- GRAFIK 2: Precision-Recall Curve per Dataset ---
fig, axes = plt.subplots(1, len(datasets), figsize=(15, 4.5), sharey=True)
for ax, fname in zip(axes, datasets):
    ds_name = fname.replace(".arff", "")
    y_t = curve_store[ds_name]["y_test"]
    for m_name, y_p in curve_store[ds_name]["models"].items():
        rec, prec = get_pr_curve_points(y_t, y_p)
        ax.plot(rec, prec, label=m_name)
    baseline_rate = y_t.mean()
    ax.axhline(baseline_rate, color="gray", linestyle="--", label=f"Random ({baseline_rate:.2f})")
    ax.set_title(f"PR Curve - {ds_name}")
    ax.set_xlabel("Recall")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
axes[0].set_ylabel("Precision")
plt.tight_layout()
plt.savefig(figures_dir / "pr_curve.png", dpi=300)
plt.close(fig)

# --- GRAFIK 3: Kurva Kumulatif Recall@20% LOC vs Random Baseline ---
fig, axes = plt.subplots(1, len(datasets), figsize=(15, 4.5), sharey=True)
for ax, fname in zip(axes, datasets):
    ds_name = fname.replace(".arff", "")
    y_t = curve_store[ds_name]["y_test"]
    loc_t = curve_store[ds_name]["loc_test"]
    for m_name, y_p in curve_store[ds_name]["models"].items():
        cum_loc, cum_def = compute_loc_recall_curve(y_t, y_p, loc_t)
        ax.plot(cum_loc, cum_def, label=m_name)
    ax.plot([0, 1], [0, 1], color="red", linestyle="--", label="Random Baseline")
    ax.axvline(0.20, color="black", linestyle=":", label="20% LOC Budget")
    ax.set_title(f"Recall vs LOC Curve - {ds_name}")
    ax.set_xlabel("Proporsi Kumulatif LOC")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
axes[0].set_ylabel("Recall (Proporsi Defect Ditemukan)")
plt.tight_layout()
plt.savefig(figures_dir / "recall_loc_curve.png", dpi=300)
plt.close(fig)

# --- GRAFIK 4: Barplot Perbandingan Recall@20% LOC ---
df_melt = df_res.melt(
    id_vars=["Dataset", "Model"],
    value_vars=["Recall20_Prob", "Recall20_Density"],
    var_name="Ranking_Strategy",
    value_name="Recall_at_20_LOC"
)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=df_melt, x="Dataset", y="Recall_at_20_LOC", hue="Ranking_Strategy", errorbar=None, ax=ax)
ax.axhline(0.20, color="red", linestyle="--", label="Random Baseline (0.20)")
ax.set_title("Recall@20% LOC: Prob Ranking vs Effort-Aware Density (Prob/LOC)")
ax.set_ylim(0, 1.0)
ax.legend()
plt.tight_layout()
plt.savefig(figures_dir / "recall20_loc_comparison.png", dpi=300)
plt.close(fig)